# Building Agents with LangGraph Series #4: Agentic Web Search

Run top to bottom. Configure credentials locally, never in notebook cells. This notebook uses public free search and National Weather Service endpoints; it does not call a paid model or Tavily. Observations are timestamped and can be stale; the comfort rule is not a travel-safety assessment.

```bash
python -m pip install openai==2.43.0 langsmith==0.14.14 langchain-core==1.4.8 langchain-openai==1.1.11 ddgs==9.16.1 requests==2.34.2 beautifulsoup4==4.15.0 python-dotenv==1.2.4 certifi==2026.10.23
```

In [ ]:
import os, json, time, uuid
from urllib.parse import urlparse
from importlib.metadata import version
from dotenv import load_dotenv
import certifi
load_dotenv()
os.environ['SSL_CERT_FILE'] = certifi.where()
print('DDGS:', version('ddgs'))
import requests
from bs4 import BeautifulSoup
session = requests.Session()
session.headers['User-Agent'] = 'ToDataBeyondEducationalTutorial/1.0 (https://todatabeyond.com)'
CITY = 'San Francisco'
LAT, LON = 37.7749, -122.4194
QUERY = f'current weather {CITY} site:weather.gov'
print('Query:', QUERY)

In [ ]:
from ddgs import DDGS
def search_web(query, max_results=3):
    if not isinstance(query, str) or not query.strip():
        raise ValueError('A nonempty search query is required')
    rows = []
    attempts = [('bing', query), ('bing', query + ' information'),
                ('duckduckgo', query), ('brave', query)]
    for backend, search_query in attempts:
        try:
            rows = DDGS(timeout=20).text(search_query, max_results=max_results, backend=backend)
            if rows: break
        except Exception:
            time.sleep(1)
    if not rows:
        raise RuntimeError('All bounded live-search attempts failed; do not invent results')
    results = []
    for row in rows:
        url = row.get('href', '')
        parsed = urlparse(url)
        if parsed.scheme == 'https' and parsed.hostname:
            results.append({'title': row.get('title', ''), 'url': url,
                            'snippet': row.get('body', '')[:240]})
    if not results:
        raise RuntimeError('Search returned no usable HTTPS sources')
    return results

sources = search_web(QUERY)
print(json.dumps(sources, indent=2, ensure_ascii=False))

In [ ]:
# Use an explicitly identified NWS forecast page, not an unrelated first result.
page_url = f'https://forecast.weather.gov/MapClick.php?lat={LAT}&lon={LON}'
response = session.get(page_url, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.content, 'html.parser')
panel = soup.select_one('#current_conditions_detail') or soup.select_one('#current-conditions')
if panel is None:
    raise RuntimeError('The NWS page layout changed; do not invent weather values')
page_text = panel.get_text(' ', strip=True)
print('Website:', page_url)
print(page_text[:900])

In [ ]:
def get_nws_json(url):
    # Only this documented public API is permitted; no arbitrary source URLs.
    if urlparse(url).scheme != 'https' or urlparse(url).hostname != 'api.weather.gov':
        raise ValueError('Expected the official NWS HTTPS API')
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

point = get_nws_json(f'https://api.weather.gov/points/{LAT},{LON}')
stations = get_nws_json(point['properties']['observationStations'])
station_id = stations['features'][0]['properties']['stationIdentifier']
observation_url = f'https://api.weather.gov/stations/{station_id}/observations/latest'
raw = get_nws_json(observation_url)
p = raw['properties']
weather = {
    'location': CITY, 'station': station_id, 'observed_at': p['timestamp'],
    'temperature': p['temperature'], 'wind_speed': p['windSpeed'],
    'conditions': p['textDescription'], 'source': observation_url,
}
print(json.dumps(weather, indent=2, ensure_ascii=False))

In [ ]:
from datetime import datetime, timezone
def comfort_check(data, now=None):
    # A teaching rule, not a travel-safety assessment. Alerts are not queried.
    now = now or datetime.now(timezone.utc)
    try:
        age = (now - datetime.fromisoformat(data['observed_at'].replace('Z', '+00:00'))).total_seconds()
        temp = data['temperature']
        value = temp['value']
    except (KeyError, TypeError, ValueError):
        return 'Unknown: incomplete observation. Check official forecasts and alerts.'
    if age < -300 or age > 10800 or temp.get('unitCode') != 'wmoUnit:degC' or value is None:
        return 'Unknown: stale, missing or unsupported temperature. Check official forecasts and alerts.'
    if not isinstance(value, (int, float)) or not -100 <= value <= 70:
        return 'Unknown: invalid temperature. Check official forecasts and alerts.'
    if value < 0 or value > 35:
        return 'Temperature comfort caution. Check official forecasts and alerts before travel.'
    return 'Temperature is within the teaching comfort range. Check official forecasts and alerts before travel.'

print(comfort_check(weather))
print('Observation units:', weather['temperature']['unitCode'], weather['wind_speed']['unitCode'])

In [ ]:
from copy import deepcopy
assert raw['properties']['timestamp'] == weather['observed_at']
assert raw['properties']['temperature'] == weather['temperature']
assert raw['properties']['windSpeed'] == weather['wind_speed']
assert all(urlparse(s['url']).scheme == 'https' for s in sources)
reference = datetime(2026, 11, 1, 12, tzinfo=timezone.utc)
fixture = {'observed_at': reference.isoformat(), 'temperature': {'unitCode': 'wmoUnit:degC', 'value': 0}}
assert 'within' in comfort_check(fixture, reference)
for value in [-1, 36]:
    changed = deepcopy(fixture); changed['temperature']['value'] = value
    assert 'caution' in comfort_check(changed, reference)
for value in [None, float('nan'), float('inf'), '20']:
    changed = deepcopy(fixture); changed['temperature']['value'] = value
    assert comfort_check(changed, reference).startswith('Unknown')
for changed in [{}, {**fixture, 'observed_at': '2020-01-01T00:00:00Z'},
                {**fixture, 'temperature': {'unitCode': 'wmoUnit:degF', 'value': 70}}]:
    assert comfort_check(changed, reference).startswith('Unknown')
try: get_nws_json('https://example.com/untrusted')
except ValueError: pass
else: raise AssertionError('Unexpected host accepted')
try: search_web('')
except ValueError: pass
else: raise AssertionError('Empty query accepted')
print('PASS: live search, HTML and official JSON; raw-field parity; units, zero, extremes, stale/missing/nonfinite values and host/input checks.')

## Original archive code and output appendix

Original blocks are retained verbatim below as nonexecuting historical source. Use the maintained examples above. Keys were placeholders; do not execute the original key-prefix print.

```text
import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import re
import time
from urllib.parse import urljoin, urlparse

# Define the city and our query
city = "San Francisco"
query = f"current weather {city} site:weather.com"

# Define a search function to get URLs
def search(query, max_results=6):
    """
    Search for URLs using DuckDuckGo

    Args:
        query (str): Search query
        max_results (int): Maximum number of results to return

    Returns:
        list: List of URLs
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))

        # Extract URLs and filter for valid ones
        urls = []
        for result in results:
            url = result.get("href") or result.get("link")
            if url and is_valid_url(url):
                urls.append(url)

        return urls

    except Exception as e:
        print(f"Search error occurred: {e}")
        # Fallback URLs
        return get_fallback_urls()

def is_valid_url(url):
    """Check if URL is valid and accessible"""
    try:
        parsed = urlparse(url)
        return bool(parsed.netloc) and bool(parsed.scheme)
    except:
        return False

def get_fallback_urls():
    """Return fallback URLs when search fails"""
    return [
        "https://weather.com/weather/today/l/USCA0987:1:US",
        "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        "https://weather.com/weather/tenday/l/USCA0987:1:US"
    ]

def fetch_weather_content(url, timeout=10):
    """
    Fetch and parse weather content from a URL

    Args:
        url (str): URL to fetch
        timeout (int): Request timeout in seconds

    Returns:
        dict: Parsed weather information
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }

        response = requests.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        # Extract weather information (this would need to be customized based on the site structure)
        weather_data = {
            'url': url,
            'title': soup.title.string if soup.title else 'No title',
            'temperature': extract_temperature(soup),
            'conditions': extract_conditions(soup),
            'raw_text': soup.get_text()[:500] + '...' if len(soup.get_text()) > 500 else soup.get_text()
        }

        return weather_data

    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return {'url': url, 'error': str(e)}
    except Exception as e:
        print(f"Error parsing {url}: {e}")
        return {'url': url, 'error': str(e)}

def extract_temperature(soup):
    """Extract temperature from parsed HTML"""
    # Common temperature selectors for weather sites
    temp_selectors = [
        '[data-testid="TemperatureValue"]',
        '.temp',
        '.temperature',
        '.current-temp',
        '[class*="temp"]'
    ]

    for selector in temp_selectors:
        temp_elem = soup.select_one(selector)
        if temp_elem:
            temp_text = temp_elem.get_text().strip()
            # Extract number and degree symbol
            temp_match = re.search(r'(-?\d+)°?[FC]?', temp_text)
            if temp_match:
                return temp_match.group(0)

    return "Temperature not found"

def extract_conditions(soup):
    """Extract weather conditions from parsed HTML"""
    condition_selectors = [
        '[data-testid="WeatherConditionsPhrase"]',
        '.condition',
        '.weather-condition',
        '.current-condition',
        '[class*="condition"]'
    ]

    for selector in condition_selectors:
        condition_elem = soup.select_one(selector)
        if condition_elem:
            return condition_elem.get_text().strip()

    return "Conditions not found"

def should_travel_today(weather_data):
    """
    Simple logic to determine if conditions are good for travel

    Args:
        weather_data (dict): Weather information

    Returns:
        str: Travel recommendation
    """
    conditions = weather_data.get('conditions', '').lower()
    temperature = weather_data.get('temperature', '')

    # Extract numeric temperature
    temp_match = re.search(r'(-?\d+)', temperature)
    temp_num = int(temp_match.group(1)) if temp_match else None

    bad_conditions = ['storm', 'heavy rain', 'snow', 'blizzard', 'tornado', 'hurricane']

    if any(bad in conditions for bad in bad_conditions):
        return "❌ Not recommended - severe weather conditions"
    elif temp_num and (temp_num < 32 or temp_num > 95):
        return f"⚠️ Use caution - extreme temperature ({temperature})"
    elif 'rain' in conditions or 'shower' in conditions:
        return "🌧️ Pack an umbrella - rainy conditions"
    else:
        return "✅ Good conditions for travel"

def main():
    """Main function to run the weather search and analysis"""
    print(f"🔍 Searching for weather information in {city}...")
    print(f"Query: {query}\n")

    # Get URLs
    urls = search(query)
    print(f"Found {len(urls)} URLs:")
    for i, url in enumerate(urls, 1):
        print(f"{i}. {url}")

    print("\n" + "="*60 + "\n")

    # Fetch and analyze weather data
    weather_results = []
    for url in urls[:3]:  # Limit to first 3 URLs
        print(f"📊 Analyzing: {url}")
        weather_data = fetch_weather_content(url)
        weather_results.append(weather_data)

        if 'error' not in weather_data:
            print(f"Temperature: {weather_data.get('temperature', 'N/A')}")
            print(f"Conditions: {weather_data.get('conditions', 'N/A')}")
            print(f"Travel recommendation: {should_travel_today(weather_data)}")
        else:
            print(f"❌ Error: {weather_data['error']}")

        print("-" * 40)
        time.sleep(1)  # Be respectful to the server

    return weather_results

# Run the updated code
if __name__ == "__main__":
    results = main()
```

```text
# Take the first URL from our search results
url = "https://www.current-news.co.uk/"

# Scrape the webpage's content
headers = {'User-Agent': 'Mozilla/5.0'}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# Let's try to extract the text from key HTML tags
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# Join the text and clean up whitespace
weather_data = "\n".join(weather_data)
weather_data = re.sub(r'\s+', ' ', weather_data)

print(f"Website: {url}\n")
print(weather_data)
```

```text
from dotenv import load_dotenv
import os
from tavily import TavilyClient

# Load environment variables and connect to the client
_ = load_dotenv()
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))
```

```text
import json
from pygments import highlight, lexers, formatters

# Run the same weather query
city = "San Francisco"
query = f"what is the current weather in {city}? Should I travel there today?"
result = client.search(query, max_results=1)

# The result content is a JSON string
data = result["results"][0]["content"]

# Parse and pretty-print the JSON to see it clearly
parsed_json = json.loads(data.replace("'", '"'))
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)
```

```text
{
    "location": {
        "name": "San Francisco",
        "region": "California",
        "country": "United States of America",
        "lat": 37.775,
        "lon": -122.4183,
        "tz_id": "America/Los_Angeles",
        "localtime_epoch": 1754546633,
        "localtime": "2025-08-06 23:03"
    },
    "current": {
        "last_updated_epoch": 1754546400,
        "last_updated": "2025-08-06 23:00",
        "temp_c": 17.8,
        "temp_f": 64.0,
        "is_day": 0,
        "condition": {
            "text": "Mist",
            "icon": "//cdn.weatherapi.com/weather/64x64/night/143.png",
            "code": 1030
        },
        "wind_mph": 6.3,
        "wind_kph": 10.1,
        "wind_degree": 263,
        "wind_dir": "W",
        "pressure_mb": 1016.0,
        "pressure_in": 30.01,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 73,
        "cloud": 25,
        "feelslike_c": 17.8,
        "feelslike_f": 64.0,
        "windchill_c": 14.0,
        "windchill_f": 57.2,
        "heatindex_c": 14.4,
        "heatindex_f": 57.9,
        "dewpoint_c": 13.7,
        "dewpoint_f": 56.6,
        "vis_km": 16.0,
        "vis_miles": 9.0,
        "uv": 0.0,
        "gust_mph": 11.5,
        "gust_kph": 18.6
    }
}
```

```text
🔍 Searching for weather information in San Francisco…
Query: current weather San Francisco site:weather.com
```

```text
Found 6 URLs:
1. https://zhidao.baidu.com/question/635787571195459284.html
2. https://www.current-news.co.uk/
3. https://zhidao.baidu.com/question/434865041797516004.html
4. https://zhidao.baidu.com/question/2279388287709892628.html
5. https://zhidao.baidu.com/question/572878914313099124.html
6. https://zhidao.baidu.com/question/2022402646620356908.html
```

```text
Website: https://www.current-news.co.uk/
```

```text
Latest Articles SolarEdge partners with Schaeffler for EV infrastructure rollout SMMT: BEV registrations rise in July despite overall car market decline ‘Fastest to date’: char.gy adds 300 chargepoints in Barnet in six weeks Fuse Energy partners with Easee for UK tariff, announces $10 million financing Next steps for UK government EV spending Next steps for UK government EV spending Speaking to EV Infrastructure News, Jon Evans, head of UK and IE market at charging platform Monta, says the government’s £63 million infrastructure investment “hits crucial infrastructure gaps”. Aug 6, 2025 Aug 5, 2025 Featured Article How vehicle-to-grid technology works: MathWorks on managing energy flow Graham Dudgeon, consultant product manager for electrical technology at engineering software provider MathWorks, speaks to EV Infrastructure News about the game-changing technology. Latest News Industry voices Event EV Infrastructure & Energy Summit 1 October — 2 October 2025 / London, UK This Summit is your comprehensive guide to navigating the complexities of EV charging and energy systems essential for driving the EV transition forward. As always, our Summit gathers the world’s leading EV experts in London to share the latest insights and case studies with a diverse audience including charge point operators, installers, manufacturers, fleet owners, local authorities, utilities, DNOs, PV and energy storage suppliers, and destination charging locations. Prepare to be inspired by our carefully curated line-up of speakers offering invaluable insights tailored to the UK market. 1 October — 2 October 2025 / London, UK This Summit is your comprehensive guide to navigating the complexities of EV charging and energy systems essential for driving the EV transition forward. As always, our Summit gathers the world’s leading EV experts in London to share the latest insights and case studies with a diverse audience including charge point operators, installers, manufacturers, fleet owners, local authorities, utilities, DNOs, PV and energy storage suppliers, and destination charging locations. Prepare to be inspired by our carefully curated line-up of speakers offering invaluable insights tailored to the UK market. Blogs Aug 5, 2025 Jul 29, 2025 Jul 25, 2025 Webinars Jun 12, 2025 Oct 21, 2024 Mar 12, 2024 Publications Oct 14, 2024 Oct 3, 2023 Oct 1, 2023 Speaking to EV Infrastructure News, Jon Evans, head of UK and IE market at charging platform Monta, says the government’s £63 million infrastructure investment “hits crucial infrastructure gaps”. Graham Dudgeon, consultant product manager for electrical technology at engineering software provider MathWorks, speaks to EV Infrastructure News about the game-changing technology. In this blog, we explore the UK’s booming used EV market and ask if it could be the key to encouraging wider uptake across the EV sector. Industry interviews and insights from the EV Infrastructure News team. Events 1–2 October 2025 | Hilton London Metropole London, UK 2 October 2025 | Hilton London Metropole London, UK Videos Now playing Now playing Copyright © 2025 All rights reserved. Informa Markets, a trading division of Informa PLC.
```